# Chapter 14 — Random Walks on Networks

Companion notebook for the [probintro textbook](https://josephausterweil.github.io/probintro/intro2/14_random_walks_networks/).

A random walk on a graph is a Markov chain whose states are nodes. We build Chibany's animal network, find its stationary distribution (π ∝ degree), sample a walk, and hand-roll PageRank on a tiny directed web.

## Setup

In [ ]:
!pip install genjax -q
print('✅ ready')

## Build the network and find π

Row-normalize the adjacency matrix into a transition matrix. The stationary distribution comes out two ways — the degree formula and power iteration — and they agree.

In [ ]:
import jax.numpy as jnp

names = ["Dog", "Hamster", "Cat", "Lion", "Tiger", "Zebra"]

# Adjacency matrix L: L[i,j] = 1 if animals i and j are connected.
L = jnp.array([
    # Dog Hamster Cat Lion Tiger Zebra
    [0, 1, 1, 0, 0, 0],   # Dog
    [1, 0, 1, 0, 0, 0],   # Hamster
    [1, 1, 0, 1, 1, 0],   # Cat   (the bridge)
    [0, 0, 1, 0, 1, 1],   # Lion
    [0, 0, 1, 1, 0, 1],   # Tiger
    [0, 0, 0, 1, 1, 0],   # Zebra
], dtype=jnp.float32)

degree = L.sum(axis=1)
P = L / degree[:, None]

pi_degree = degree / degree.sum()

def power_iterate(v, P, steps):
    for _ in range(steps):
        v = v @ P
    return v
pi_power = power_iterate(jnp.ones(6) / 6, P, 200)

for i, name in enumerate(names):
    print(f"{name:8s} degree {int(degree[i])}   "
          f"pi(degree) {pi_degree[i]:.3f}   pi(power) {pi_power[i]:.3f}")

## Sample a walk (GenJAX)

The walk is a Markov chain — sample it with `categorical` on the log of the current row of `P`.

In [ ]:
import jax
import jax.random as jr
from genjax import gen, categorical

LOGP = jnp.log(P)

def make_walk(n_steps):
    @gen
    def walk(start):
        node = start
        visited = [node]
        for t in range(n_steps):
            node = categorical(LOGP[node]) @ f"v_{t}"
            visited.append(node)
        return jnp.array(visited)
    return walk

walk8 = make_walk(8)
seq = walk8.simulate(jr.key(0), (1,)).get_retval()   # start at Hamster (index 1)
print(" -> ".join(names[int(i)] for i in seq))

## Confirm the degree law by sampling

One long walk; tally where it spends its time. Visit frequency tracks degree share.

In [ ]:
def run_long(key, start, n):
    def step(node, k):
        nxt = jr.categorical(k, LOGP[node])
        return nxt, nxt
    _, visited = jax.lax.scan(step, start, jr.split(key, n))
    return visited

visited = run_long(jr.key(2), 2, 20000)
freq = jnp.array([jnp.mean((visited == i).astype(float)) for i in range(6)])
for i, name in enumerate(names):
    print(f"{name:8s} visited {float(freq[i]):.2f}   (degree share {float(pi_degree[i]):.2f})")

## Hand-rolled PageRank

For a directed graph the degree shortcut fails, so find π by power iteration — with the ε-teleport (Ch 13) that keeps the walk ergodic.

In [ ]:
pages = ["A", "B", "C", "D"]
M = jnp.array([
    # A  B  C  D
    [0, 1, 1, 0],   # A -> B, C
    [0, 0, 1, 0],   # B -> C
    [1, 0, 0, 0],   # C -> A
    [0, 0, 1, 0],   # D -> C
], dtype=jnp.float32)

n = 4
out_links = M.sum(axis=1, keepdims=True)
P_links = M / out_links

epsilon = 0.15
uniform = jnp.ones((n, n)) / n
P_surfer = (1 - epsilon) * P_links + epsilon * uniform

pagerank = power_iterate(jnp.ones(n) / n, P_surfer, 200)
for i, page in enumerate(pages):
    print(f"page {page}: PageRank {float(pagerank[i]):.3f}")

## Try it yourself

1. Add an edge (e.g. Dog–Zebra) to `L` and recompute `pi_degree`. Which nodes gain, which lose?
2. Make the walk directed (set `L[i,j]=1` but `L[j,i]=0`). Does `pi_degree` still match `pi_power`? Why not?
3. In PageRank, try `epsilon = 0.01` and `0.5`. How does the ranking shift? What does `epsilon = 1` mean?